## Étape 4.1 : lecture des données

Le CSV est lu depuis HDFS. On vérifie le nombre de colonnes (24 attendues) et de lignes (environ 200 000 attendues).

In [1]:
import os
from pyspark.sql import SparkSession

DATA_PATH = os.environ.get(
    "DATA_PATH",
    "hdfs://localhost:9000/user/miguel/projet_3IABD/fraudTest.csv",
)

OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("fraude")
    .master("local[2]")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# Lecture du CSV depuis HDFS.
raw = spark.read.csv(DATA_PATH, header=True, inferSchema=False)

# Contrôles : 23 colonnes attendues (dont l'index _c0), environ 555 719 lignes attendues
print("Nombre de colonnes :", len(raw.columns))
print("Nombre de lignes :", raw.count())

raw.printSchema()

26/11/19 17:58:45 WARN Utils: Your hostname, LAPTOP-2O6PEQV6 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/19 17:58:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 17:58:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Nombre de colonnes : 23
Nombre de lignes : 555719
root
 |-- _c0: string (nullable = true)
 |-- trans_date_trans_time: string (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: string (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- long: string (nullable = true)
 |-- city_pop: string (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: string (nullable = true)
 |-- merch_lat: string (nullable = true)
 |-- merch_long: string (nullable = true)
 |-- is_fraud: string (nullable = true)



## Étape 4.2 : renommage et typage des colonnes

Les colonnes sont renommées en français, puis typées : les montants, l'âge et la cible en nombres, la date en type date, le reste en texte.

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, LongType

# Correspondance entre les noms (la colonne d'index "_c0" est ignorée automatiquement)
RENOMMAGE = {
    "trans_date_trans_time": "date_heure_transaction",
    "cc_num": "num_carte",
    "merchant": "marchand",
    "category": "categorie_marchand",
    "amt": "montant",
    "first": "prenom_client",
    "last": "nom_client",
    "gender": "genre",
    "street": "rue_client",
    "city": "ville",
    "state": "etat",
    "zip": "code_postal",
    "lat": "lat_client",
    "long": "long_client",
    "city_pop": "population_ville",
    "job": "profession",
    "dob": "date_naissance",
    "trans_num": "id_transaction",
    "unix_time": "timestamp_unix",
    "merch_lat": "lat_marchand",
    "merch_long": "long_marchand",
    "is_fraud": "est_fraude",
}

# Contrôle : toutes les colonnes attendues doivent exister dans le fichier.
manquantes = [c for c in RENOMMAGE if c not in raw.columns]
if manquantes:
    raise ValueError(f"Colonnes absentes du fichier : {manquantes}")

df = raw.select([F.col(c).alias(RENOMMAGE[c]) for c in RENOMMAGE])

# Typage des colonnes
df = (
    df
    .withColumn("date_heure_transaction", F.to_timestamp(F.col("date_heure_transaction"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("date_naissance", F.to_date(F.col("date_naissance"), "yyyy-MM-dd"))
    .withColumn("montant", F.col("montant").cast(DoubleType()))
    .withColumn("lat_client", F.col("lat_client").cast(DoubleType()))
    .withColumn("long_client", F.col("long_client").cast(DoubleType()))
    .withColumn("lat_marchand", F.col("lat_marchand").cast(DoubleType()))
    .withColumn("long_marchand", F.col("long_marchand").cast(DoubleType()))
    .withColumn("population_ville", F.col("population_ville").cast(IntegerType()))
    .withColumn("timestamp_unix", F.col("timestamp_unix").cast(LongType()))
    .withColumn("est_fraude", F.col("est_fraude").cast(IntegerType()))
)

# Contrôle : une valeur qui ne se convertit pas devient null.
cols_typees = [
    "date_heure_transaction", "date_naissance", "montant",
    "lat_client", "long_client", "lat_marchand", "long_marchand",
    "population_ville", "timestamp_unix", "est_fraude"
]
for c in cols_typees:
    n = df.filter(F.col(c).isNull()).count()
    print(f"{c} : {n} valeur(s) non convertie(s)")

df.printSchema()

date_heure_transaction : 0 valeur(s) non convertie(s)


date_naissance : 0 valeur(s) non convertie(s)


montant : 0 valeur(s) non convertie(s)


lat_client : 0 valeur(s) non convertie(s)


long_client : 0 valeur(s) non convertie(s)


lat_marchand : 0 valeur(s) non convertie(s)


long_marchand : 0 valeur(s) non convertie(s)


population_ville : 0 valeur(s) non convertie(s)


timestamp_unix : 0 valeur(s) non convertie(s)
est_fraude : 0 valeur(s) non convertie(s)
root
 |-- date_heure_transaction: timestamp (nullable = true)
 |-- num_carte: string (nullable = true)
 |-- marchand: string (nullable = true)
 |-- categorie_marchand: string (nullable = true)
 |-- montant: double (nullable = true)
 |-- prenom_client: string (nullable = true)
 |-- nom_client: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- rue_client: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- etat: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- lat_client: double (nullable = true)
 |-- long_client: double (nullable = true)
 |-- population_ville: integer (nullable = true)
 |-- profession: string (nullable = true)
 |-- date_naissance: date (nullable = true)
 |-- id_transaction: string (nullable = true)
 |-- timestamp_unix: long (nullable = true)
 |-- lat_marchand: double (nullable = true)
 |-- long_marchand: double (nullable = true)

## Étape 4.3 : statistiques descriptives

On décrit les colonnes numériques, la répartition de la cible `est_fraude`, puis les liens entre les variables et la fraude. Lors d'une analyse préalable sur le fichier complet, le taux de fraude était d'environ 5 % et les corrélations proches de 0. Ces chiffres devront être retrouvés ici, et interprétés après exécution.

In [3]:
from pyspark.sql import functions as F

colonnes_num = ["montant", "population_ville", "est_fraude"]

df.select(colonnes_num).describe().show()

# 1. Corrélation de chaque variable numérique avec la cible
for c in ["montant", "population_ville"]:
    r = df.stat.corr(c, "est_fraude")
    print(f"corrélation {c} / est_fraude : {r:.4f}")

# 2. Nombre de transactions normales et frauduleuses
total = df.count()
(
    df.groupBy("est_fraude").count()
    .withColumn("taux_%", F.round(F.col("count") / total * 100, 2))
    .orderBy("est_fraude")
    .show()
)

# 3. Montant moyen par classe
df.groupBy("est_fraude").agg(F.round(F.avg("montant"), 2).alias("montant_moyen")).show()

# 4. Taux de fraude par catégorie
for c in ["categorie_marchand", "genre"]:
    (
        df.groupBy(c)
        .agg(F.round(F.avg("est_fraude") * 100, 2).alias("taux_fraude_%"))
        .orderBy(F.desc("taux_fraude_%"))
        .show()
    )

# 5. Table croisée pour le Heatmap : taux de fraude (%) par catégorie et par heure
heatmap_df = (
    df.withColumn("heure", F.hour("date_heure_transaction"))
    .groupBy("categorie_marchand")
    .pivot("heure", list(range(24)))
    .agg(F.round(F.avg("est_fraude") * 100, 2))
    .fillna(0.0)
)
heatmap_df.toPandas().to_csv(f"{OUT}/heatmap_fraude.csv", index=False)
print("Heatmap exporté dans", f"{OUT}/heatmap_fraude.csv")

# 6. Export des statistiques descriptives
stats_pd = df.select(colonnes_num).describe().toPandas()
stats_pd.to_csv(f"{OUT}/statistiques.csv", index=False)
print("Statistiques exportées dans", f"{OUT}/statistiques.csv")

+-------+-----------------+------------------+--------------------+
|summary|          montant|  population_ville|          est_fraude|
+-------+-----------------+------------------+--------------------+
|  count|           555719|            555719|              555719|
|   mean|69.39281023322941| 88221.88791817447|0.003859864427885...|
| stddev|156.7459413553154|300390.89206099225| 0.06200784461178418|
|    min|              1.0|                23|                   0|
|    max|         22768.11|           2906700|                   1|
+-------+-----------------+------------------+--------------------+



corrélation montant / est_fraude : 0.1823


corrélation population_ville / est_fraude : -0.0049


+----------+------+------+
|est_fraude| count|taux_%|
+----------+------+------+
|         0|553574| 99.61|
|         1|  2145|  0.39|
+----------+------+------+



+----------+-------------+
|est_fraude|montant_moyen|
+----------+-------------+
|         1|       528.36|
|         0|        67.61|
+----------+-------------+



+------------------+-------------+
|categorie_marchand|taux_fraude_%|
+------------------+-------------+
|      shopping_net|         1.21|
|          misc_net|         0.98|
|       grocery_pos|         0.92|
|      shopping_pos|         0.43|
|     gas_transport|         0.27|
|            travel|         0.23|
|          misc_pos|         0.21|
|       grocery_net|         0.21|
|     personal_care|         0.18|
|     entertainment|         0.15|
|       food_dining|         0.14|
|    health_fitness|         0.14|
|              home|         0.13|
|         kids_pets|         0.13|
+------------------+-------------+



+-----+-------------+
|genre|taux_fraude_%|
+-----+-------------+
|    M|         0.39|
|    F|         0.38|
+-----+-------------+



26/11/19 17:59:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Heatmap exporté dans outputs/heatmap_fraude.csv


Statistiques exportées dans outputs/statistiques.csv


## Étape 4.4 : nettoyage

Le nettoyage porte sur cinq points : suppression de la colonne constante `devise`, retrait des données personnelles, doublons sur l'identifiant de transaction, valeurs manquantes, et plages de valeurs. Lors de l'analyse préalable, aucun doublon ni valeur manquante n'a été trouvé : la cellule le vérifie et le documente.

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os

# 1. Nettoyage de base (sans supprimer encore les infos client)
df_prof = df.drop("timestamp_unix").dropDuplicates(["id_transaction"]).dropna()
df_prof = df_prof.filter(
    (F.col("montant") > 0)
    & (F.col("population_ville") > 0)
    & F.col("lat_client").between(-90, 90)
    & F.col("long_client").between(-180, 180)
    & F.col("est_fraude").isin(0, 1)
)

# 2. Création d'un identifiant de profil basé sur les données personnelles
df_prof = (
    df_prof
    .withColumn("id_profil", F.concat_ws("_", "prenom_client", "nom_client", "date_naissance"))
    .withColumn("heure", F.hour("date_heure_transaction"))
    .withColumn("ts_sec", F.unix_timestamp("date_heure_transaction"))
)

# 3. Fenêtres temporelles par profil (strictement tournées vers le PASSÉ pour éviter le data leakage)
w_passe = Window.partitionBy("id_profil").orderBy("ts_sec").rowsBetween(Window.unboundedPreceding, -1)
w_ordre = Window.partitionBy("id_profil").orderBy("ts_sec")
w_24h = Window.partitionBy("id_profil").orderBy("ts_sec").rangeBetween(-86400, -1)  # 24h précédentes

# 4. Variables comportementales : comparaison transaction actuelle vs habitudes du client
df_prof = (
    df_prof
    # Montant moyen historique du client (si 1ère transaction, on prend le montant actuel)
    .withColumn("montant_moyen_client", F.coalesce(F.avg("montant").over(w_passe), F.col("montant")))
    # Ratio : dépense-t-il 5x ou 10x plus que d'habitude ?
    .withColumn("ratio_montant_habituel", F.col("montant") / (F.col("montant_moyen_client") + 1.0))
    # Écart par rapport à l'heure habituelle du client
    .withColumn("heure_moyenne_client", F.coalesce(F.avg("heure").over(w_passe), F.col("heure")))
    .withColumn("ecart_heure_habituelle", F.abs(F.col("heure") - F.col("heure_moyenne_client")))
    # Fréquence : nombre de transactions de ce client dans les dernières 24h
    .withColumn("nb_trans_24h", F.count("*").over(w_24h))
    # Vitesse : minutes écoulées depuis la transaction précédente de ce client
    .withColumn(
        "delai_derniere_trans_min",
        F.coalesce((F.col("ts_sec") - F.lag("ts_sec", 1).over(w_ordre)) / 60.0, F.lit(9999.0))
    )
)

# Contrôle visuel : voir un exemple de profil avant anonymisation
df_prof.select(
    "prenom_client", "nom_client", "montant", "montant_moyen_client",
    "ratio_montant_habituel", "nb_trans_24h", "est_fraude"
).filter(F.col("est_fraude") == 1)

# 5. Anonymisation : maintenant que le profil est calculé, on supprime les données personnelles !
df_prof = df_prof.drop("prenom_client", "nom_client", "rue_client", "num_carte", "id_transaction", "id_profil", "ts_sec")

parquet_path = f"file://{os.path.abspath(OUT)}/df_prof.parquet"
df_prof.write.mode("overwrite").parquet(parquet_path)
df_prof = spark.read.parquet(parquet_path)
print("Lignes prêtes avec profils :", df_prof.count())

Lignes prêtes avec profils : 555719


## Étape 4.5 : features

Les modèles ne lisent que des nombres. Cette étape extrait l'heure et le jour de la semaine, encode les colonnes texte, puis regroupe toutes les variables dans un vecteur `features`. Seules les colonnes texte à peu de valeurs distinctes sont retenues ; les colonnes à très nombreuses valeurs (ville, état, agence, localisation, appareil de transaction) sont écartées pour garder un modèle simple.

In [5]:
from pyspark.sql.types import IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark import StorageLevel

# 1. Feature engineering complémentaire
df_feat = (
    df_prof
    .withColumn("jour_semaine", F.dayofweek(F.col("date_heure_transaction")))
    .withColumn(
        "age",
        F.floor(F.datediff(F.col("date_heure_transaction"), F.col("date_naissance")) / 365.25).cast(IntegerType())
    )
    .withColumn(
        "distance_km",
        F.sqrt(
            F.pow(F.col("lat_client") - F.col("lat_marchand"), 2)
            + F.pow(F.col("long_client") - F.col("long_marchand"), 2)
        ) * 111.0
    )
)

# 2. Poids pour le déséquilibre
stats = df_feat.agg(F.count("*").alias("total"), F.sum("est_fraude").alias("fraudes")).first()
poids_fraude = (stats["total"] - stats["fraudes"]) / stats["fraudes"]

df_feat = df_feat.withColumn(
    "poids",
    F.when(F.col("est_fraude") == 1, F.lit(poids_fraude)).otherwise(F.lit(1.0))
)

# 3. Séparation Train / Test
train_raw, test_raw = df_feat.randomSplit([0.8, 0.2], seed=42)

# 4. Variables (on ajoute les nouvelles variables de profil comportemental)
cols_cat = ["categorie_marchand", "genre", "etat"]
cols_num = [
    "montant", "population_ville", "age", "heure", "jour_semaine", "distance_km",
    "montant_moyen_client", "ratio_montant_habituel",
    "ecart_heure_habituelle", "nb_trans_24h", "delai_derniere_trans_min"
]

# 5. Pipeline
indexeurs = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cols_cat]
encodeurs = [OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_ohe"]) for c in cols_cat]

entrees = [f"{c}_ohe" for c in cols_cat] + cols_num
assembleur = VectorAssembler(inputCols=entrees, outputCol="features_raw")
normalisateur = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)

pipeline = Pipeline(stages=indexeurs + encodeurs + [assembleur, normalisateur])
modele_prep = pipeline.fit(train_raw)

train_df = modele_prep.transform(train_raw).select("features", "est_fraude", "poids").persist(StorageLevel.MEMORY_AND_DISK)
test_df = modele_prep.transform(test_raw).select("features", "est_fraude", "poids").persist(StorageLevel.MEMORY_AND_DISK)

print(f"Train : {train_df.count()} | Test : {test_df.count()}")

Train : 444341 | Test : 111378


## Étape 4.6 : Machine Learning


.

In [7]:
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# 1. Taille exacte du vecteur d'entrée (calculée automatiquement)
nb_features = train_df.first()["features"].size
print("Nombre de features en entrée du PMC :", nb_features)

# 2. Rééquilibrage du jeu d'entraînement UNIQUEMENT (le MLP ne supporte pas weightCol)
# On garde 100% des fraudes et 3% des transactions normales (~11% de fraude en train)
train_mlp = train_df.sampleBy("est_fraude", fractions={0: 0.03, 1: 1.0}, seed=42)
print("Lignes utilisées pour entraîner le PMC :", train_mlp.count())

# 3. Architecture du Perceptron Multicouches : Entrée -> 32 neurones -> 16 neurones -> 2 classes
layers = [nb_features, 32, 16, 2]

mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="est_fraude",
    layers=layers,
    maxIter=50,
    blockSize=128,
    seed=42
)

# 4. Entraînement et évaluation sur le MÊME test_df non modifié
print("\n=== Entraînement : Perceptron Multicouches (MLP) ===")
modele_mlp = mlp.fit(train_mlp)
preds_mlp = modele_mlp.transform(test_df)

eval_pr = BinaryClassificationEvaluator(labelCol="est_fraude", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
eval_roc = BinaryClassificationEvaluator(labelCol="est_fraude", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
eval_f1 = MulticlassClassificationEvaluator(labelCol="est_fraude", predictionCol="prediction", metricName="fMeasureByLabel", metricLabel=1.0)

auprc = eval_pr.evaluate(preds_mlp)
auroc = eval_roc.evaluate(preds_mlp)
f1_fraude = eval_f1.evaluate(preds_mlp)

print(f"AUPRC : {auprc:.4f} | AUROC : {auroc:.4f} | F1_Fraude : {f1_fraude:.4f}")
print("Matrice de confusion :")
preds_mlp.groupBy("est_fraude", "prediction").count().orderBy("est_fraude", "prediction").show()

Nombre de features en entrée du PMC : 77
Lignes utilisées pour entraîner le PMC : 15022

=== Entraînement : Perceptron Multicouches (MLP) ===
AUPRC : 0.6478 | AUROC : 0.9870 | F1_Fraude : 0.3561
Matrice de confusion :
+----------+----------+------+
|est_fraude|prediction| count|
+----------+----------+------+
|         0|       0.0|109702|
|         0|       1.0|  1246|
|         1|       0.0|    67|
|         1|       1.0|   363|
+----------+----------+------+

